In [2]:
import os
import json
import subprocess
from collections import defaultdict

In [9]:
with open('../../cleaned_dataset.json') as f:
    dataset = json.load(f)

In [12]:
class_count = defaultdict(int)

for k, entry in dataset.items():
    for c in entry['classes']:
        class_count[c] += 1

class_count

defaultdict(int,
            {'toxic': 22881,
             'other-functional': 9206,
             'anti-bacterial': 31427,
             'anti-cancer': 12986,
             'anti-fungal': 13300,
             'anti-parasitic': 6723,
             'anti-viral': 7517,
             'metabolic': 13487,
             'cell-cell-communication': 3352,
             'inhibitor': 3538,
             'signal-peptide': 22650,
             'immunological': 4572,
             'drug-delivery': 2578})

In [17]:
with open("../../non_functional_sequences.txt", "r") as f:
    non_functional_sequences = f.read().splitlines()

non_functional_sequences = [s.strip() for s in non_functional_sequences if s.strip() != '']


In [18]:
# common sequs
len(set(dataset.keys()).intersection(set(non_functional_sequences)))

0

In [19]:
# combining all sequences 
id_seq_map = dict()

for seq in dataset.keys():
    id = f"{len(id_seq_map)}_functional"
    id_seq_map[id] = seq

for seq in non_functional_sequences:
    id = f"{len(id_seq_map)}_nonfunct"
    id_seq_map[id] = seq

In [20]:
len(id_seq_map)

280830

In [48]:
input_fasta = "sequences.fasta"
with open(input_fasta, "w") as f:
    for i, seq in id_seq_map.items():
        # Header format: >Index
        f.write(f">{i}_id\n{seq}\n")

In [88]:
cmd = "cd-hit -i sequences.fasta -o nr_sequences.fasta -c 0.7 -n 5 -l 5 -d 0"
subprocess.run(cmd, shell=True, check=True)

print("CD-HIT clustering complete.")

Program: CD-HIT, V4.8.1 (+OpenMP), Nov 12 2024, 10:35:24
Command: cd-hit -i sequences.fasta -o nr_sequences.fasta -c
         0.7 -n 5 -l 5 -d 0

Started: Wed Jun 24 10:58:12 2026
                            Output                              
----------------------------------------------------------------
Discarding invalid sequence or sequence without identifier and description!

>5190_functional_id
TLE-K-TLE-A-TLE-A-TLE-A
Discarding invalid sequence or sequence without identifier and description!

>5254_functional_id
FFHLHFH-DCΔDOPA
Discarding invalid sequence or sequence without identifier and description!

>5263_functional_id
ARG-ARG-H
Discarding invalid sequence or sequence without identifier and description!

>5280_functional_id
KKWΒ2,2WKK
Discarding invalid sequence or sequence without identifier and description!

>5327_functional_id
A(ABU)F(ABU)LPGGGGVA(ABU)L(ABU)(DHA)EAI
Discarding invalid sequence or sequence without identifier and description!

>5333_functional_id
DM-RDYG

In [89]:
# reading clusters
clusters = []
with open("nr_sequences.fasta.clstr") as f:
    for line in f:
        if line.startswith(">"):
            clusters.append([])
        else:
            id = line.split('>')[1].split('_id')[0].strip()
            if '*' in line: val = 200
            else : val = float(line.split(' at')[1].split('%')[0])
            
            seq = id_seq_map[id]
            if 'nonfunct' in line:
                classes = set(['non-functional'])
            else:
                classes = dataset[seq]['classes']
                
            clusters[-1].append([id, val, classes])



In [90]:
mapped_ids = set()

for cluster in clusters:
    for seq in cluster:
        mapped_ids.add(seq[0])



In [91]:
len(set(id_seq_map)- mapped_ids)

6618

In [92]:
# saving clusters data
data = []

for cluster in clusters:
    cluster_data = []
    for e in cluster:
        cluster_data.append(
            {
                'sequence': id_seq_map[e[0]],
                'classes': list(e[2])
            }
        )
    data.append(cluster_data)

with open("clustered_dataset_7.json", "w") as f:
    json.dump(data, f, indent=4)
